In [48]:
# IMPORTS
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import joblib
import matplotlib.pyplot as plt 

In [37]:
# raw data
pi_data = pd.read_csv("data/pima_indian.csv")
pi_data['Gender'] = '1' # all patients are female from pima indian dataset; female = 1, male = 0
cdcd_data = pd.read_csv("data/cdcd.csv")
cdcd_data['gender'] = cdcd_data['gender'].map({'Male': 0, 'Female': 1})
cdcd_data = cdcd_data.dropna(subset=['gender']) # drop any where there is no gender value

# only get the columns that will be used: gender, age, bmi, glucose, outcome
pi_data = pi_data[['Gender', 'Age', 'BMI', 'Glucose', 'Outcome']]
cdcd_data = cdcd_data[['gender','age','bmi','blood_glucose_level','diabetes']]
# rename pi_data columns to match cdcd_data
pi_data.rename(columns={'Gender':'gender', 'Age':'age', 'BMI':'bmi','Glucose':'blood_glucose_level','Outcome':'diabetes'}, inplace=True)

print(cdcd_data['gender'].unique())
pi_data.describe()
#cdcd_data.describe()

[1. 0.]


,age,bmi,blood_glucose_level,diabetes
count,768.000000,768.000000,768.000000,768.000000
mean,33.240885,31.992578,120.894531,0.348958
std,11.760232,7.884160,31.972618,0.476951
min,21.000000,0.000000,0.000000,0.000000
25%,24.000000,27.300000,99.000000,0.000000
50%,29.000000,32.000000,117.000000,0.000000
75%,41.000000,36.600000,140.250000,1.000000
max,81.000000,67.100000,199.000000,1.000000


In [38]:
print("Duplicates (pi):", pi_data.duplicated().sum()) 
print("Duplicates: (cdcd)", cdcd_data.duplicated().sum()) 

# no duplicate data :)
print(pi_data.isnull().sum())       # counts of missing values
print((cdcd_data == 0).sum())         # counts of zeros
print((pi_data == 0).sum())         # counts of zeros

Duplicates (pi): 1
Duplicates: (cdcd) 23422
gender                 0
age                    0
bmi                    0
blood_glucose_level    0
diabetes               0
dtype: int64
gender                 41430
age                        0
bmi                        0
blood_glucose_level        0
diabetes               91482
dtype: int64
gender                   0
age                      0
bmi                     11
blood_glucose_level      5
diabetes               500
dtype: int64


In [39]:
# clean data

# cannot have 0 in these columns
cols_to_fix = ["bmi", "blood_glucose_level"]
for col in cols_to_fix:
    pi_data[col] = pi_data[col].replace(0, pi_data[col].median())

# limit cdcd data
cdcd_data = cdcd_data[cdcd_data['age'] >= 18] # adults only
cdcd_data = cdcd_data[(cdcd_data['bmi'] >= 15) & (cdcd_data['bmi'] <= 60)] # bmi should be 15-60
cdcd_data = cdcd_data[(cdcd_data['blood_glucose_level'] >= 50) & (cdcd_data['blood_glucose_level'] <= 250)] # glucose level should be 50-250 

# drop duplicate data
pi_data = pi_data.drop_duplicates()
cdcd_data = cdcd_data.drop_duplicates()

#print(pi_data.describe())
#print(cdcd_data.describe())
pi_data.to_csv("data/pima_indian_clean.csv", index=False)
cdcd_data.to_csv("data/cdcd_clean.csv", index=False)

# combine datasets
combined_clean = pd.concat([pi_data, cdcd_data], ignore_index=True)
combined_clean.to_csv("data/combined_clean.csv", index=False)
combined_clean.describe()

,age,bmi,blood_glucose_level,diabetes
count,60895.000000,60895.000000,60895.000000,60895.000000
mean,48.375942,28.952581,135.782462,0.099745
std,17.807898,6.694477,35.877906,0.299663
min,18.000000,15.010000,44.000000,0.000000
25%,33.000000,24.250000,100.000000,0.000000
50%,48.000000,27.600000,140.000000,0.000000
75%,62.000000,32.400000,159.000000,0.000000
max,81.000000,67.100000,240.000000,1.000000


In [40]:
# data
X = combined_clean[["gender","age", "bmi", "blood_glucose_level"]] # features
y = combined_clean["diabetes"] # target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


print("X_train Shape:",  X_train.shape)
print("X_test Shape:", X_test.shape)
print("Y_train Shape:", y_train.shape)
print("Y_test Shape:", y_test.shape)

X_train Shape: (42626, 4)
X_test Shape: (18269, 4)
Y_train Shape: (42626,)
Y_test Shape: (18269,)


In [68]:
# train logistic regression
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced')
log_reg.fit(X_train, y_train)

# test logistic regression
y_pred_lr = log_reg.predict(X_test)
lr_acc = accuracy_score(y_test, y_pred_lr)

In [69]:
# train random forest
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

# test random forest
y_pred_rf = rf.predict(X_test)
rf_acc = accuracy_score(y_test, y_pred_rf)


In [55]:
# save models
joblib.dump(log_reg, 'models/logreg_model.joblib')
joblib.dump(rf, 'models/rf_model.joblib')

['models/rf_model.joblib']

In [70]:
# evaluation
# confusion matrix: tn, fp, fn, tp
print("logistic regression accuracy:", lr_acc)
# accuracy: 0.912
cm_lr = confusion_matrix(y_test, y_pred_lr)
print(cm_lr)

print("random forest accuracy", rf_acc)
# accuracy: 0.901
cm_rf = confusion_matrix(y_test, y_pred_rf)
print(cm_rf)


logistic regression accuracy: 0.7372598390716514
[[12133  4367]
 [  433  1336]]
random forest accuracy 0.9001040013137008
[[15971   529]
 [ 1296   473]]
